# DANDI 001695 HighDensityCrossBrain → NeuroPy pipeline

Loads locally downloaded [DANDI 001695 — High-density Hippocampal-Cortical dynamics](https://dandiarchive.org/dandiset/001695) NWB files from `HighDensityCrossBrain/001695` via registered `DANDI001695NWBDataSessionFormat` / `DataSessionLoader.dandi_nwb_001695_session`, then runs the NeuroPy pipeline (flattened spikes, position QC, sleep-state epochs, placefields, decoding).

**Workflow:**
1. **Initialize session** — load NWB → write export cache under `HighDensityCrossBrain/export/001695/{M01}/{ses-id}/`
2. **POSTLOAD** — activity epochs, laps, PBE/non-PBE
3. **Pipeline** — `NeuropyPipeline` + computations; saves `basedir/loadedSessPickle.pkl`

**Paths:** `sess.basepath` is the subject folder under `001695/`; export cache is under `HighDensityCrossBrain/export/001695/{M01}/{ses-id}/`.

**Install:** `uv sync --all-extras --python 3.10`

**Note:** The ecephys-only NWB (`2024-03-08`) has no position data. Use a `behavior+ecephys` file via `NWB_FILENAME`.

### Session Info
Seems to only have one epoch named 'MAZE3' which is a linear track
I see the animal correctly running laps on the linear track though

In [1]:
%config IPCompleter.use_jedi = False
# %xmode Verbose
# %xmode context
%pdb off
%load_ext autoreload
%autoreload 3

import sys
from pathlib import Path

# required to enable non-blocking interaction:
%gui qt5

import importlib
from copy import deepcopy
from numba import jit
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
# pd.options.mode.dtype_backend = 'pyarrow' # use new pyarrow backend instead of numpy
from attrs import define, field, fields, Factory, make_class
import tables as tb
from datetime import datetime, timedelta

# Pho's Formatting Preferences
import builtins

import IPython
from IPython.core.formatters import PlainTextFormatter
from IPython import get_ipython

from pyphocorehelpers.preferences_helpers import set_pho_preferences, set_pho_preferences_concise, set_pho_preferences_verbose
set_pho_preferences_concise()
# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
from pyphocorehelpers.gui.Jupyter.AsyncExecutionHelper import run_async

# BEGIN PPRINT CUSTOMIZATION ___________________________________________________________________________________________ #

## IPython pprint
from pyphocorehelpers.pprint import wide_pprint, wide_pprint_ipython, wide_pprint_jupyter, MAX_LINE_LENGTH
# Override default pprint
builtins.pprint = wide_pprint

ip = get_ipython()

from pyphocorehelpers.ipython_helpers import CustomFormatterMagics

# Register the magic
get_ipython().register_magics(CustomFormatterMagics)

text_formatter: PlainTextFormatter = ip.display_formatter.formatters['text/plain']
text_formatter.max_width = MAX_LINE_LENGTH
text_formatter.for_type(object, wide_pprint_jupyter)


# END PPRINT CUSTOMIZATION ___________________________________________________________________________________________ #

from pyphocorehelpers.print_helpers import get_now_time_str, get_now_day_str
from pyphocorehelpers.indexing_helpers import get_dict_subset

## Pho's Custom Libraries:
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path, file_uri_from_path
from pyphocorehelpers.Filesystem.open_in_system_file_manager import reveal_in_system_file_manager
import pyphocorehelpers.programming_helpers as programming_helpers

# NeuroPy (Diba Lab Python Repo) Loading
# from neuropy import core
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from typing_extensions import TypeAlias
from nptyping import NDArray
import neuropy.utils.type_aliases as types

from neuropy.analyses.placefields import PlacefieldComputationParameters
from neuropy.core.epoch import NamedTimerange, Epoch
from neuropy.core.ratemap import Ratemap
from neuropy.core.session.Formats.BaseDataSessionFormats import DataSessionFormatRegistryHolder, DataSessionFormatBaseRegisteredClass
from neuropy.core.session.Formats.BaseDataSessionFormats import HardcodedProcessingParameters
from neuropy.core.session.Formats.Specific.NWBDataSessionFormat import NWBDataSessionFormatRegisteredClass

from neuropy.utils.matplotlib_helpers import matplotlib_file_only, matplotlib_configuration, matplotlib_configuration_update
from neuropy.core.neuron_identities import NeuronIdentityTable, neuronTypesList, neuronTypesEnum
from neuropy.utils.mixins.AttrsClassHelpers import AttrsBasedClassHelperMixin, serialized_field, serialized_attribute_field, non_serialized_field, custom_define
from neuropy.utils.mixins.HDF5_representable import HDF_DeserializationMixin, post_deserialize, HDF_SerializationMixin, HDFMixin, HDF_Converter

## For computation parameters:
from neuropy.analyses.placefields import PlacefieldComputationParameters
from neuropy.utils.dynamic_container import DynamicContainer
from neuropy.utils.result_context import IdentifyingContext
from neuropy.core.session.Formats.BaseDataSessionFormats import find_local_session_paths
from neuropy.core.user_annotations import UserAnnotationsManager

from pyphocorehelpers.print_helpers import print_object_memory_usage, print_dataframe_memory_usage, print_value_overview_only, DocumentationFilePrinter, print_keys_if_possible, generate_html_string, document_active_variables
from pyphocorehelpers.programming_helpers import metadata_attributes
from pyphocorehelpers.function_helpers import function_attributes
## Pho Programming Helpers:
from pyphocorehelpers.print_helpers import DocumentationFilePrinter, TypePrintMode, print_keys_if_possible, debug_dump_object_member_shapes, print_value_overview_only, document_active_variables
from pyphocorehelpers.programming_helpers import IPythonHelpers, PythonDictionaryDefinitionFormat, MemoryManagement, inspect_callable_arguments, get_arguments_as_optional_dict, GeneratedClassDefinitionType, CodeConversion
from pyphocorehelpers.notebook_helpers import NotebookCellExecutionLogger
from pyphocorehelpers.gui.Qt.TopLevelWindowHelper import TopLevelWindowHelper, print_widget_hierarchy
from pyphocorehelpers.indexing_helpers import reorder_columns, reorder_columns_relative, dict_to_full_array
from pyphocorehelpers.DataStructure.RenderPlots.MatplotLibRenderPlots import MatplotlibRenderPlots

# pyPhoPlaceCellAnalysis:
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import NeuropyPipeline # get_neuron_identities
from pyphoplacecellanalysis.General.Mixins.ExportHelpers import export_pyqtgraph_plot
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_load_session, batch_extended_computations, batch_evaluate_required_computations
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import PipelineSavingScheme # used in perform_pipeline_save
from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import PipelineJupyterHelpers, CustomProcessingPhases
from pyphocorehelpers.assertion_helpers import Assert

import pyphoplacecellanalysis.External.pyqtgraph as pg

from pyphocorehelpers.exception_helpers import ExceptionPrintingContext, CapturedException
from pyphoplacecellanalysis.General.Batch.NonInteractiveProcessing import batch_perform_all_plots
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.LongShortTrackComputations import JonathanFiringRateAnalysisResult
from pyphoplacecellanalysis.General.Mixins.CrossComputationComparisonHelpers import _find_any_context_neurons
from pyphoplacecellanalysis.General.Batch.runBatch import BatchSessionCompletionHandler # for `post_compute_validate(...)`
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BasePositionDecoder
from pyphoplacecellanalysis.SpecificResults.AcrossSessionResults import AcrossSessionsResults
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.SpikeAnalysis import SpikeRateTrends # for `_perform_long_short_instantaneous_spike_rate_groups_analysis`
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.LongShortTrackComputations import SingleBarResult, InstantaneousSpikeRateGroupsComputation, TruncationCheckingResults # for `BatchSessionCompletionHandler`, `AcrossSessionsAggregator`
from pyphoplacecellanalysis.General.Mixins.CrossComputationComparisonHelpers import SplitPartitionMembership
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import DirectionalPlacefieldGlobalComputationFunctions, DirectionalLapsResult, TrackTemplates, DecoderDecodedEpochsResult
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.RankOrderComputations import RankOrderGlobalComputationFunctions,  RankOrderComputationsContainer, RankOrderResult, RankOrderAnalyses
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.MultiContextComputationFunctions.DirectionalPlacefieldGlobalComputationFunctions import TrackTemplates
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.ComputationFunctionRegistryHolder import ComputationFunctionRegistryHolder, computation_precidence_specifying_function, global_function
from pyphocorehelpers.Filesystem.path_helpers import set_posix_windows
from pyphoplacecellanalysis.Analysis.Decoder.reconstruction import BasePositionDecoder, DecodedFilterEpochsResult, SingleEpochDecodedResult

from pyphocorehelpers.assertion_helpers import Assert

# Plotting
# import pylustrator # customization of figures
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
_bak_rcParams = mpl.rcParams.copy()

matplotlib.use('Qt5Agg')
# %matplotlib inline
# %matplotlib auto

# _restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')
_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')

import seaborn as sns

# import pylustrator # call `pylustrator.start()` before creating your first figure in code.
from pyphoplacecellanalysis.Pho2D.matplotlib.visualize_heatmap import visualize_heatmap, visualize_heatmap_pyqtgraph # used in `plot_kourosh_activity_style_figure`
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.SpikeRasters import plot_multiple_raster_plot, plot_raster_plot
from pyphoplacecellanalysis.General.Mixins.DataSeriesColorHelpers import UnitColoringMode, DataSeriesColorHelpers
from pyphoplacecellanalysis.General.Pipeline.Stages.DisplayFunctions.SpikeRasters import _build_default_tick, build_scatter_plot_kwargs
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.Mixins.Render2DScrollWindowPlot import Render2DScrollWindowPlotMixin, ScatterItemData
from pyphoplacecellanalysis.General.Pipeline.Stages.ComputationFunctions.SpikeAnalysis import SpikeRateTrends
from pyphoplacecellanalysis.General.Mixins.SpikesRenderingBaseMixin import SpikeEmphasisState
from pyphoplacecellanalysis.General.Model.SpecificComputationParameterTypes import ComputationKWargParameters
from pyphoplacecellanalysis.SpecificResults.PhoDiba2023Paper import PAPER_FIGURE_figure_1_add_replay_epoch_rasters, PAPER_FIGURE_figure_1_full, PAPER_FIGURE_figure_3, main_complete_figure_generations
# from pyphoplacecellanalysis.SpecificResults.fourthYearPresentation import *

# Jupyter Widget Interactive
import ipywidgets as widgets
from IPython.display import display, HTML
from pyphocorehelpers.Filesystem.open_in_system_file_manager import reveal_in_system_file_manager
from pyphoplacecellanalysis.GUI.IPyWidgets.pipeline_ipywidgets import interactive_pipeline_widget, interactive_pipeline_files
from pyphocorehelpers.gui.Jupyter.simple_widgets import fullwidth_path_widget, render_colors

from datetime import datetime, date, timedelta
from pyphocorehelpers.print_helpers import get_now_day_str, get_now_rounded_time_str


known_data_session_type_properties_dict = DataSessionFormatRegistryHolder.get_registry_known_data_session_type_dict()
active_data_session_types_registered_classes_dict = DataSessionFormatRegistryHolder.get_registry_data_session_type_class_name_dict()

DAY_DATE_STR: str = date.today().strftime("%Y-%m-%d")
DAY_DATE_TO_USE = f'{DAY_DATE_STR}' # used for filenames throught the notebook
print(f'DAY_DATE_STR: {DAY_DATE_STR}, DAY_DATE_TO_USE: {DAY_DATE_TO_USE}')

NOW_DATETIME: str = get_now_rounded_time_str()
NOW_DATETIME_TO_USE = f'{NOW_DATETIME}' # used for filenames throught the notebook
print(f'NOW_DATETIME: {NOW_DATETIME}, NOW_DATETIME_TO_USE: {NOW_DATETIME_TO_USE}')

def get_global_variable(var_name):
    """ used by `PipelineJupyterHelpers._build_pipeline_custom_processing_mode_selector_widget(...)` to update the notebook's variables """
    return globals()[var_name]
    
def update_global_variable(var_name, value):
    """ used by `PipelineJupyterHelpers._build_pipeline_custom_processing_mode_selector_widget(...)` to update the notebook's variables """
    globals()[var_name] = value

from pyphocorehelpers.gui.Jupyter.simple_widgets import build_global_data_root_parent_path_selection_widget
all_paths = [Path(r'H:\Data'), Path(r'I:\Data'), Path(r'/home/halechr/FastData'), Path('/Volumes/SwapSSD/Data'), Path('/Users/pho/data'), Path(r'/media/halechr/MAX/Data'), Path(r'W:\Data'), Path(r'/home/halechr/cloud/turbo/Data'), Path(r'/Volumes/MoverNew/data'), Path(r'/home/halechr/turbo/Data'), Path(r'/Users/pho/cloud/turbo/Data')] # Path('/Volumes/FedoraSSD/FastData'), 
global_data_root_parent_path = None
def on_user_update_path_selection(new_path: Path):
    global global_data_root_parent_path
    new_global_data_root_parent_path = new_path.resolve()
    global_data_root_parent_path = new_global_data_root_parent_path
    print(f'global_data_root_parent_path changed to {global_data_root_parent_path}')
    assert global_data_root_parent_path.exists(), f"global_data_root_parent_path: {global_data_root_parent_path} does not exist! Is the right computer's config commented out above?"
            
global_data_root_parent_path_widget = build_global_data_root_parent_path_selection_widget(all_paths, on_user_update_path_selection)
global_data_root_parent_path_widget


Automatic pdb calling has been turned OFF


H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\utils\mixins\time_slicing.py:405: UserWarning: registration of accessor <class 'neuropy.utils.mixins.time_slicing.TimePointEventAccessor'> under name 'time_point_event' for type <class 'pandas.core.frame.DataFrame'> is overriding a preexisting attribute with the same name.
  class TimePointEventAccessor(TimeColumnAliasesProtocol, TimeSlicableObjectProtocol, DataframeMetadataProtocol):
h:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\.venv\lib\site-packages\replay_trajectory_classification\likelihoods\multiunit_likelihood.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
h:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\.venv\lib\site-packages\hdf5storage\utilities.py:44: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_reso

field.name: "merged_directional_placefields", variable_name: "merged_directional_placefields"
field.name: "rank_order_shuffle_analysis", variable_name: "rank_order_shuffle_analysis"
field.name: "directional_decoders_decode_continuous", variable_name: "directional_decoders_decode_continuous"
field.name: "directional_decoders_evaluate_epochs", variable_name: "directional_decoders_evaluate_epochs"
field.name: "directional_decoders_epoch_heuristic_scoring", variable_name: "directional_decoders_epoch_heuristic_scoring"
field.name: "directional_train_test_split", variable_name: "directional_train_test_split"
field.name: "long_short_decoding_analyses", variable_name: "long_short_decoding_analyses"
field.name: "long_short_rate_remapping", variable_name: "long_short_rate_remapping"
field.name: "long_short_inst_spike_rate_groups", variable_name: "long_short_inst_spike_rate_groups"
field.name: "wcorr_shuffle_analysis", variable_name: "wcorr_shuffle_analysis"
field.name: "non_pbe_epochs_results", 

ToggleButtons(description='Data Root:', layout=Layout(width='auto'), options=(WindowsPath('H:/Data'), WindowsPath('I:/Data'), WindowsPath('W:/Data')), style=ToggleButtonsStyle(button_width='max-content'), tooltip='global_data_root_parent_path', value=WindowsPath('H:/Data'))

In [2]:
from pynwb import NWBHDF5IO
from neuropy.core.session.data_session_loader import DataSessionLoader
from neuropy.core.session.Formats.BaseDataSessionFormats import DataSessionFormatRegistryHolder
from neuropy.core.session.Formats.Specific.DANDI001695NWBDataSessionFormat import DANDI001695NWBDataSessionFormatRegisteredClass
from pyphoplacecellanalysis.General.Pipeline.NeuropyPipeline import NeuropyPipeline, PipelineSavingScheme
from pyphoplacecellanalysis.SpecificResults.PendingNotebookCode import final_process_bapun_all_comps, build_proper_epoch_intervals

In [3]:
REPO_ROOT = Path(r"H:\Data\DANDI\HighDensityCrossBrain").resolve()
DATA_ROOT = REPO_ROOT / '001695'
EXPORT_ROOT = REPO_ROOT / 'export'
# SUBJECT = 'M01'
# NWB_FILENAME = 'sub-M01_ses-20240312T100000_behavior+ecephys.nwb'
# NWB_FILENAME = 'sub-M01_ses-20240313T100000_behavior+ecephys.nwb'


SUBJECT = 'M02'
NWB_FILENAME = 'sub-M02_ses-20240318T100000_behavior+ecephys.nwb'

SESSION_BASEDIR = DATA_ROOT / f'sub-{SUBJECT}'



UNIT_LOCATION_FILTER = 'CA1'
PLOT_SUBSAMPLE = 25
active_data_mode_name = 'dandi_nwb_001695'
basedir = SESSION_BASEDIR.resolve()
# force_reload = False
force_reload = True
saving_mode = PipelineSavingScheme.TEMP_THEN_OVERWRITE

override_parameters_flat_keypaths_dict = {
    'nwb.nwb_filename': NWB_FILENAME,
    'nwb.unit_location_filter': UNIT_LOCATION_FILTER,
    'nwb.export_root': str(EXPORT_ROOT),
    'preprocessing.epoch_estimation_parameters.laps.use_direction_dependent_laps': False,
}

assert DATA_ROOT.is_dir(), f'Missing data root: {DATA_ROOT}'
assert SESSION_BASEDIR.is_dir(), f'Missing subject dir: {SESSION_BASEDIR}'
print('REPO_ROOT:', REPO_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('SESSION_BASEDIR:', SESSION_BASEDIR)
print('NWB_FILENAME:', NWB_FILENAME)
print('active_data_mode_name:', active_data_mode_name)
print('export cache root:', EXPORT_ROOT / '001695' / SUBJECT)

REPO_ROOT: H:\Data\DANDI\HighDensityCrossBrain
DATA_ROOT: H:\Data\DANDI\HighDensityCrossBrain\001695
SESSION_BASEDIR: H:\Data\DANDI\HighDensityCrossBrain\001695\sub-M02
NWB_FILENAME: sub-M02_ses-20240318T100000_behavior+ecephys.nwb
active_data_mode_name: dandi_nwb_001695
export cache root: H:\Data\DANDI\HighDensityCrossBrain\export\001695\M02


In [4]:
nwb_files = sorted(SESSION_BASEDIR.glob('*.nwb'))
print(f'Found {len(nwb_files)} NWB files under {SESSION_BASEDIR}')
for p in nwb_files:
    tag = ' (ecephys-only: no position)' if 'behavior' not in p.name else ''
    print(' -', p.name, tag)

nwb_path = DANDI001695NWBDataSessionFormatRegisteredClass.find_nwb_file(SESSION_BASEDIR, nwb_filename=NWB_FILENAME)
print('Selected NWB:', nwb_path)

Found 6 NWB files under H:\Data\DANDI\HighDensityCrossBrain\001695\sub-M02
 - sub-M02_ses-20240226T100000_ecephys.nwb  (ecephys-only: no position)
 - sub-M02_ses-20240307T100000_ecephys.nwb  (ecephys-only: no position)
 - sub-M02_ses-20240312T100000_behavior+ecephys.nwb 
 - sub-M02_ses-20240313T100000_behavior+ecephys.nwb 
 - sub-M02_ses-20240314T100000_behavior+ecephys.nwb 
 - sub-M02_ses-20240318T100000_behavior+ecephys.nwb 
Selected NWB: H:\Data\DANDI\HighDensityCrossBrain\001695\sub-M02\sub-M02_ses-20240318T100000_behavior+ecephys.nwb


In [5]:
with NWBHDF5IO(str(nwb_path), mode='r') as io:
    nwbf = io.read()
    print('session_description:', nwbf.session_description)
    if nwbf.intervals and 'SleepStates' in nwbf.intervals:
        ss = nwbf.intervals['SleepStates']
        print('SleepStates:', len(ss), '| cols:', list(ss.colnames))
        display(ss.to_dataframe().head())
    if nwbf.processing and 'behavior' in nwbf.processing and 'AnimalPosition' in nwbf.processing['behavior'].data_interfaces:
        pos_iface = nwbf.processing['behavior']['AnimalPosition']
        pos_series = pos_iface.spatial_series['Position'] if 'Position' in pos_iface.spatial_series else list(pos_iface.spatial_series.values())[0]
        print('position shape:', pos_series.data.shape)
    else:
        print('No behavior position in this NWB file.')
    udf = nwbf.units.to_dataframe()
    print('units:', len(udf), '| cell_area counts:\n', udf['cell_area'].value_counts())

session_description: Novel Maze Session with Pre and Post sleep
SleepStates: 2068 | cols: ['start_time', 'stop_time', 'state']


,start_time,stop_time,state
id,,,
0,1.0000,899.0000,WAKE
1,389.6800,389.7088,Ripple
2,410.9632,410.9984,Ripple
3,452.4240,452.4760,Ripple
4,452.6024,452.6272,Ripple


position shape: (67722, 2)
units: 150 | cell_area counts:
 CA1    98
RSC    27
CA3    25
Name: cell_area, Length: 3, dtype: int64


## Initialize session from NWB

First load builds NeuroPy export cache (`.npy` under `export/001695/M01/ses-…/`). Re-running uses the cache unless you delete those files or change `NWB_FILENAME`.

In [6]:
sess = DataSessionLoader.dandi_nwb_001695_session(SESSION_BASEDIR, override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict)
print('context:', sess.get_context())
print('neurons:', sess.neurons.n_neurons, '| position samples:', len(sess.position.time))
print('epochs:', sess.epochs.get_unique_labels().tolist())
print('ripples:', len(sess.ripple.to_dataframe()) if sess.ripple is not None else 0)
print('filePrefix:', sess.filePrefix)
for cache_suffix in ['.neurons.npy', '.position.npy', '.paradigm.npy', '.flattened.spikes.npy', '.interpolated_spike_positions.npy']:
    cache_path = sess.filePrefix.with_suffix(cache_suffix)
    print(f'  {cache_path.name}:', 'OK' if cache_path.exists() else 'MISSING', '->', cache_path)
sess

	 Loading success: .interpolated_spike_positions.npy.
externally computed ripple_df.pkl not found. Falling back to .ripple.npy...
Failure loading .ripple.npy. Must recompute.

computing ripple epochs for session...

Computation failed with error 'NoneType' object has no attribute 'get_signal'. Skipping .ripple
Loading success: .mua.npy.
Loading success: .pbe.npy.
Loading success: .non_pbe.npy.
Computing spikes_df PBEs column results : "spikes_df"... encounter KeyError 't_rel_seconds' when attempting to access spk_df using its spk_df.spikes.time_variable_name variable. Original spk_df.spikes.time_variable_name: "t_rel_seconds". Changing it to "t_seconds" and proceeding forward


	 time variable changed from 't_rel_seconds' to 't_seconds'.


	 time variable changed!
done.
Computing added spike scISI column results : "spikes_df"... done.
context: dandi_nwb_001695_M02_001695_ses-20240318T100000
neurons: 98 | position samples: 67722
epochs: ['WAKE0', 'Ripple0', 'Ripple1', 'Ripple2', 'Ripple3', 'Ripple4', 'Ripple5', 'Ripple6', 'Ripple7', 'Ripple8', 'Ripple9', 'Ripple10', 'Ripple11', 'Ripple12', 'Ripple13', 'Ripple14', 'Ripple15', 'Ripple16', 'Ripple17', 'Ripple18', 'Ripple19', 'Ripple20', 'Ripple21', 'Ripple22', 'Ripple23', 'Ripple24', 'Ripple25', 'Ripple26', 'Ripple27', 'Ripple28', 'Ripple29', 'Ripple30', 'Ripple31', 'Ripple32', 'Ripple33', 'Ripple34', 'Ripple35', 'Ripple36', 'NREM0', 'Ripple37', 'Ripple38', 'Ripple39', 'Ripple40', 'Ripple41', 'Ripple42', 'Ripple43', 'Ripple44', 'WAKE1', 'Ripple45', 'Ripple46', 'Ripple47', 'NREM1', 'Ripple48', 'Ripple49', 'Ripple50', 'Ripple51', 'Ripple52', 'Ripple53', 'Ripple54', 'WAKE2', 'Ripple55', 'Ripple56', 'Ripple57', 'Ripple58', 'Ripple59', 'Ripple60', 'Ripple61', 'Ripple62', 'Ripple6

DataSession(configured from manual recinfo: DynamicContainer({'source_file': None, 'channel_groups': None, 'skipped_channels': None, 'discarded_channels': None, 'n_channels': None, 'dat_sampling_rate': None, 'eeg_sampling_rate': None}))

In [7]:
DANDI001695NWBDataSessionFormatRegisteredClass.POSTLOAD_estimate_laps_and_replays(sess)
print('epochs after POSTLOAD:', sess.epochs.get_unique_labels().tolist())
print('laps:', len(sess.laps.to_dataframe()) if sess.laps is not None else 0)
sess.epochs.to_dataframe().head(10)

POSTLOAD_estimate_laps_and_replays()...
fixing up DANDI 001695 session computation epochs...
	done. new epochs: 
2069 epochs
array([[0.9965, 898.996],
       [389.677, 389.705],
       [410.96, 410.995],
       ...,
       [11528.2, 11528.2],
       [11538.6, 11538.6],
       [11541.9, 11542]])


computing PBE epochs for session...

computing estimated replay epochs for session...

	 using KnownFilterEpochs.PBE as surrogate replays...


H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\core\session\Formats\Specific\DANDI001695NWBDataSessionFormat.py:597: UserWarning: Could not estimate replays for DANDI 001695 session dandi_nwb_001695_M02_001695_ses-20240318T100000: 'DataFrame' object has no attribute 'to_dataframe'
  warnings.warn(f'Could not estimate replays for DANDI 001695 session {sess.get_context()}: {e}')


computing non_PBE epochs for session...



H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\core\epoch.py:2188: UserWarning: curr_epochs already empty prior to any filtering
  warn(f'curr_epochs already empty prior to any filtering')


Saving non_pbe results results : "H:/Data/DANDI/HighDensityCrossBrain/export/M02/ses-20240318T100000.non_pbe.npy"... ses-20240318T100000.non_pbe.npy saved
done.


DataSession(configured from manual recinfo: DynamicContainer({'source_file': None, 'channel_groups': None, 'skipped_channels': None, 'discarded_channels': None, 'n_channels': None, 'dat_sampling_rate': None, 'eeg_sampling_rate': None}))

epochs after POSTLOAD: ['WAKE0', 'Ripple0', 'Ripple1', 'Ripple2', 'Ripple3', 'Ripple4', 'Ripple5', 'Ripple6', 'Ripple7', 'Ripple8', 'Ripple9', 'Ripple10', 'Ripple11', 'Ripple12', 'Ripple13', 'Ripple14', 'Ripple15', 'Ripple16', 'Ripple17', 'Ripple18', 'Ripple19', 'Ripple20', 'Ripple21', 'Ripple22', 'Ripple23', 'Ripple24', 'Ripple25', 'Ripple26', 'Ripple27', 'Ripple28', 'Ripple29', 'Ripple30', 'Ripple31', 'Ripple32', 'Ripple33', 'Ripple34', 'Ripple35', 'Ripple36', 'NREM0', 'Ripple37', 'Ripple38', 'Ripple39', 'Ripple40', 'Ripple41', 'Ripple42', 'Ripple43', 'Ripple44', 'WAKE1', 'Ripple45', 'Ripple46', 'Ripple47', 'NREM1', 'Ripple48', 'Ripple49', 'Ripple50', 'Ripple51', 'Ripple52', 'Ripple53', 'Ripple54', 'WAKE2', 'Ripple55', 'Ripple56', 'Ripple57', 'Ripple58', 'Ripple59', 'Ripple60', 'Ripple61', 'Ripple62', 'Ripple63', 'Ripple64', 'Ripple65', 'Ripple66', 'Ripple67', 'Ripple68', 'Ripple69', 'NREM2', 'Ripple70', 'Ripple71', 'Ripple72', 'Ripple73', 'Ripple74', 'Ripple75', 'Ripple76', 'Ripple7

,start,stop,label,duration
0,0.9965,898.9965,WAKE0,898.0000
1,389.6765,389.7053,Ripple0,0.0288
2,410.9597,410.9949,Ripple1,0.0352
3,452.4205,452.4725,Ripple2,0.0520
4,452.5989,452.6237,Ripple3,0.0248
5,452.8605,452.8933,Ripple4,0.0328
6,453.7525,453.7885,Ripple5,0.0360
7,458.0829,458.1061,Ripple6,0.0232
8,464.1189,464.1453,Ripple7,0.0264
9,464.5317,464.5645,Ripple8,0.0328


## Load NeuroPy pipeline

Wraps the initialized session for downstream computations. Saves `basedir/loadedSessPickle.pkl` after a successful fresh load.

In [8]:
# known_data_session_type_properties_dict = DataSessionFormatRegistryHolder.get_registry_known_data_session_type_dict()
known_data_session_type_properties_dict = DataSessionFormatRegistryHolder.get_registry_known_data_session_type_dict(
    override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict
)
active_data_session_types_registered_classes_dict = DataSessionFormatRegistryHolder.get_registry_data_session_type_class_name_dict()
active_data_mode_type_properties = known_data_session_type_properties_dict[active_data_mode_name]

print(f'basedir: {basedir} | force_reload: {force_reload}')

override_parameters_flat_keypaths_dict = (override_parameters_flat_keypaths_dict or {}) | {'nwb.nwb_filename': NWB_FILENAME}
curr_active_pipeline = NeuropyPipeline.try_init_from_saved_pickle_or_reload_if_needed(active_data_mode_name, active_data_mode_type_properties, override_basepath=Path(basedir), force_reload=force_reload, override_parameters_flat_keypaths_dict=override_parameters_flat_keypaths_dict, skip_save_on_initial_load=False)
print('pipeline session context:', curr_active_pipeline.sess.get_context())
print('pickle path:', basedir / 'loadedSessPickle.pkl')

basedir: H:\Data\DANDI\HighDensityCrossBrain\001695\sub-M02 | force_reload: True
Skipping loading from pickled file because force_reload == True.
build_logger(full_logger_string="2026-07-09_11-07-15.Apogee.dandi_nwb_001695_pipeline.dandi_nwb_001695", file_logging_dir: None):
	 Loading success: .interpolated_spike_positions.npy.
externally computed ripple_df.pkl not found. Falling back to .ripple.npy...
Failure loading .ripple.npy. Must recompute.

computing ripple epochs for session...

Computation failed with error 'NoneType' object has no attribute 'get_signal'. Skipping .ripple
Loading success: .mua.npy.
Loading success: .pbe.npy.
Loading success: .non_pbe.npy.
Computing spikes_df PBEs column results : "spikes_df"... done.
Computing added spike scISI column results : "spikes_df"... done.
POSTLOAD_estimate_laps_and_replays()...
fixing up DANDI 001695 session computation epochs...
	done. new epochs: 
2069 epochs
array([[0.9965, 898.996],
       [389.677, 389.705],
       [410.96, 410.

H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\core\session\Formats\Specific\DANDI001695NWBDataSessionFormat.py:597: UserWarning: Could not estimate replays for DANDI 001695 session dandi_nwb_001695_M02_001695_ses-20240318T100000: 'DataFrame' object has no attribute 'to_dataframe'
  warnings.warn(f'Could not estimate replays for DANDI 001695 session {sess.get_context()}: {e}')


computing non_PBE epochs for session...



H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\NeuroPy\neuropy\core\epoch.py:2188: UserWarning: curr_epochs already empty prior to any filtering
  warn(f'curr_epochs already empty prior to any filtering')


Saving non_pbe results results : "H:/Data/DANDI/HighDensityCrossBrain/export/M02/ses-20240318T100000.non_pbe.npy"... ses-20240318T100000.non_pbe.npy saved
done.
Updated preprocessing parameter: preprocessing.epoch_estimation_parameters.laps.use_direction_dependent_laps -> epoch_estimation_parameters.laps.use_direction_dependent_laps = False
WARN: PipelineWithComputedPipelineStageMixin.update_parameters(...): too early to set the computation_config override_parameters, not yet at the computation stage!!
  - Computation parameters can only be updated after the pipeline reaches the computed stage.
  - Preprocessing parameters can be updated at any stage.
Saving (file mode 'w+b') pickle file results : "H:/Data/DANDI/HighDensityCrossBrain/001695/sub-M02/loadedSessPickle.pkl"... 	moving new output at 'H:\Data\DANDI\HighDensityCrossBrain\001695\sub-M02\20260709115920-loadedSessPickle.pkltmp' -> to desired location: 'H:\Data\DANDI\HighDensityCrossBrain\001695\sub-M02\loadedSessPickle.pkl'
	Sav

In [9]:
pos_df = curr_active_pipeline.sess.position.to_dataframe().position.compute_speed_info()
print('position duration (s):', pos_df['t'].max() - pos_df['t'].min())
print('x range:', pos_df['x'].min(), pos_df['x'].max())
print('y range:', pos_df['y'].min(), pos_df['y'].max())
pos_df.iloc[::PLOT_SUBSAMPLE].plot.scatter(x='x', y='y', c='speed', s=4, cmap='viridis', figsize=(6, 5))

position duration (s): 2827.8629555555553
x range: 6.1052452188391895 116.69142245734311
y range: 10.40262848127758 20.942789475717543


<Axes: xlabel='x', ylabel='y'>

In [10]:
DANDI001695NWBDataSessionFormatRegisteredClass.session_fixup_epochs(curr_active_pipeline.sess, enable_global_epoch=True)
curr_active_pipeline = final_process_bapun_all_comps(curr_active_pipeline=curr_active_pipeline, active_data_mode_name=active_data_mode_name, posthoc_save=False, time_bin_size=0.500, overwrite_extant=False, fail_on_exception=True, active_computation_functions_name_includelist=['pf_computation', 'pfdt_computation', 'position_decoding'])

WARN: already fixedup session epochs.
	restoring backed up epochs:
	done. new epochs: 
2069 epochs
array([[0.9965, 898.996],
       [389.677, 389.705],
       [410.96, 410.995],
       ...,
       [11528.2, 11528.2],
       [11538.6, 11538.6],
       [11541.9, 11542]])




           start        stop       label  duration
0         0.9965    898.9965       WAKE0  898.0000
1       389.6765    389.7053     Ripple0    0.0288
2       410.9597    410.9949     Ripple1    0.0352
3       452.4205    452.4725     Ripple2    0.0520
4       452.5989    452.6237     Ripple3    0.0248
5       452.8605    452.8933     Ripple4    0.0328
...          ...         ...         ...       ...
2062  11307.3981  11307.4381  Ripple2022    0.0400
2063  11475.0717  11475.1325  Ripple2023    0.0608
2064  11493.8013  11493.8349  Ripple2024    0.0336
2065  11528.1765  11528.2069  Ripple2025    0.0304
2066  11538.6109  11538.6469  Ripple2026    0.0360
2067  11541.9445  11541.9805  Ripple2027    0.0360

[2069 rows x 4 columns]

hardcoded_params.decoder_building_session_names: ['WAKE0', 'activity_GLOBAL']
hardcoded_params.non_global_activity_session_names: ['WAKE0']
active_data_session_types_registered_classes_dict: {'dandi_nwb': <class 'neuropy.core.session.Formats.Specific.NWBDataSessionFormat.NWBDataSessionFormatRegisteredClass'>, 'kdiba': <class 'neuropy.core.session.Formats.Specific.KDibaOldDataSessionFormat.KDibaOldDataSessionFormatRegisteredClass'>, 'dandi_nwb_001695': <class 'neuropy.core.session.Formats.Specific.DANDI001695NWBDataSessionFormat.DANDI001695NWBDataSessionFormatRegisteredClass'>, 'bapun': <class 'neuropy.core.session.Formats.Specific.BapunDataSessionFormat.BapunDataSessionFormatRegisteredClass'>, 'rachel': <class 'neuropy.core.session.Formats.Specific.RachelDataSessionFormat.RachelDataSessionFormat'>}
WARN: already fixedup session epochs.
	done. new epochs: 
2069 epochs
array([[0.9965, 898.996],
       [389.677, 389.705],
       [410.96, 410.995],
       ...,
       [11528.2, 11528.2],
  

	 no change in time_variable_name. It will remain t_seconds.


Applying session filter named "WAKE11"...


	 no change in time_variable_name. It will remain t_seconds.


	hardcoded_params.grid_bin_bounds: ((0.0, 100.0), (0.0, 100.0))
beginning compute...
i: 0, active_epoch_names: ['WAKE11']
updating computation_results...
done.
Performing perform_action_for_all_contexts with action EvaluationActions.EVALUATE_COMPUTATIONS on filtered_session with filter named "WAKE11"...
curr_active_computation_params.pf_params.computation_epochs: 1 epochs
array([[4601, 8269]])

due to includelist, including only 3 out of 18 registered computation functions.
Performing _execute_computation_functions(...) with 3 registered_computation_functions...
Recomputing active_epoch_placefields1D... 	 done.
Recomputing active_epoch_placefields2D... 	 done.
Recomputing active_epoch_time_dependent_placefields... 	 done.
Recomputing active_epoch_time_dependent_placefields2D... 	 done.
_execute_computation_functions(...): 
	accumulated_errors: None
	computation_times: {'_perform_baseline_placefield_computation': datetime.datetime(2026, 7, 9, 11, 59, 38, 420143), '_perform_position_deco

In [ ]:
curr_active_pipeline.save_pipeline(active_pickle_filename=basedir / 'loadedSessPickle.pkl', saving_mode=saving_mode)
print('Saved pipeline pickle to', basedir / 'loadedSessPickle.pkl')
print('Session export cache remains at', curr_active_pipeline.sess.filePrefix.parent)

finalized_loaded_sess_pickle_path: H:\Data\DANDI\HighDensityCrossBrain\001695\sub-M02\loadedSessPickle.pkl
Saving (file mode 'w+b') pickle file results : "H:/Data/DANDI/HighDensityCrossBrain/001695/sub-M02/20260709115957-loadedSessPickle.pkl"... 

# 🎨 2024-02-06 - Other Plotting

In [ ]:
from pyphoplacecellanalysis.Pho2D.PyQtPlots.TimeSynchronizedPlotters.TimeSynchronizedPlacefieldsPlotter import TimeSynchronizedPlacefieldsPlotter

_restore_previous_matplotlib_settings_callback = matplotlib_configuration_update(is_interactive=True, backend='Qt5Agg')

#  Create a new `SpikeRaster2D` instance using `_display_spike_raster_pyqtplot_2D` and capture its outputs:
curr_active_pipeline.reload_default_display_functions()
curr_active_pipeline.prepare_for_display()

## `LauncherWidget`: GUI

In [ ]:
from pyphoplacecellanalysis.General.Pipeline.Stages.Display import DisplayFunctionItem
from pyphocorehelpers.gui.Qt.tree_helpers import find_tree_item_by_text
from pyphoplacecellanalysis.GUI.Qt.MainApplicationWindows.LauncherWidget.LauncherWidget import LauncherWidget

widget = LauncherWidget()
treeWidget = widget.mainTreeWidget # QTreeWidget
widget.build_for_pipeline(curr_active_pipeline=curr_active_pipeline)
widget.show()

## ✅ 2025-09-19 - Clean programmmatic figure outputs 

In [ ]:
from pyphocorehelpers.plotting.figure_management import PhoActiveFigureManager2D, capture_new_figures_decorator
fig_man = PhoActiveFigureManager2D(name=f'fig_man') # Initialize a new figure manager
from pyphoplacecellanalysis.GUI.PyQtPlot.Widgets.DockAreaWrapper import DockAreaWrapper
from pyphoplacecellanalysis.General.Mixins.ExportHelpers import programmatic_render_to_file, programmatic_display_to_PDF, extract_figures_from_display_function_output
from neuropy.core.session.Formats.BaseDataSessionFormats import HardcodedProcessingParameters
from neuropy.core.session.Formats.Specific.NWBDataSessionFormat import NWBDataSessionFormatRegisteredClass


In [ ]:

hardcoded_params: HardcodedProcessingParameters = NWBDataSessionFormatRegisteredClass._get_session_specific_parameters(session_context=curr_active_pipeline.get_session_context())
# hardcoded_params

# hardcoded_params.decoder_building_session_names
# hardcoded_params.non_global_activity_session_names

fig_man.close_all()

# subset_includelist = ['maze1', 'maze2', 'maze_GLOBAL'] # Day5TwoNovel
# subset_includelist = ['roam', 'sprinkle'] # Day4

In [ ]:

# subset_includelist = hardcoded_params.decoder_building_session_names
subset_includelist = None
print(f'subset_includelist: {subset_includelist}')

In [ ]:
display_fn_kwargs = dict(subplots=(None, 9),
    fig_column_width=None,   # key fix — uses data aspect ratio for width
    fig_row_height=1.0,
    resolution_multiplier=1.0,
)

# display_fn_kwargs = dict(subplots=(None, 5))

# _out = dict()
# _out['_display_2d_placefield_result_plot_ratemaps_2D'] = curr_active_pipeline.display(display_function='_display_2d_placefield_result_plot_ratemaps_2D', active_session_configuration_context=IdentifyingContext(format_name='bapun',animal='RatS',session_name='Day5TwoNovel',filter_name='maze1'), **display_fn_kwargs) # _display_2d_placefield_result_plot_ratemaps_2D
# _out['_display_2d_placefield_result_plot_ratemaps_2D'] = curr_active_pipeline.display(display_function='_display_2d_placefield_result_plot_ratemaps_2D', active_session_configuration_context=IdentifyingContext(format_name='bapun',animal='RatS',session_name='Day5TwoNovel',filter_name='maze2'), **display_fn_kwargs) # _display_2d_placefield_result_plot_ratemaps_2D


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_2d_placefield_result_plot_ratemaps_2D', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True, **display_fn_kwargs)


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_2d_placefield_occupancy', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True)


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_1d_placefields', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True, **display_fn_kwargs)


In [ ]:
_out_list = programmatic_render_to_file(curr_active_pipeline=curr_active_pipeline, curr_display_function_name='_display_1d_placefield_validations', subset_includelist=subset_includelist, 
                                        write_vector_format=True, write_png=True, debug_print=True, **display_fn_kwargs)